
# Bài tập Python (ML/DL): ViT, CLIP, SOM, GNN, GCN

**Sinh viên:** *(điền tên bạn)*  
**Môn:** Python / Machine Learning  
**Ngày thực hiện:** *(dd/mm/yyyy)*

> File này là một notebook chuẩn, chia thành 5 chủ đề ứng dụng thực tế:
> 9) ViT (Vision Transformer) – *phân loại ảnh / kiểm tra chất lượng ảnh (demo)*  
> 10) CLIP – *tìm ảnh theo văn bản (image search) / zero-shot classification (demo)*  
> 11) Self-Organizing Maps (SOM) – *phân cụm khách hàng (Iris)*  
> 12) GNN (Graph Neural Networks) – *phát hiện cộng đồng trên mạng xã hội (Karate Club)*  
> 13) GCN (Graph Convolutional Networks) – *phân loại node bán giám sát (Karate Club)*

⚠️ **Phụ thuộc**: Notebook ưu tiên code thuần `numpy`, `scikit-learn`, `networkx` để chạy được ngoại tuyến.  
Một số cell **tùy chọn** sẽ cài `torch`, `transformers`, `timm` để chạy ViT/CLIP pre-trained nếu bạn có internet.


In [2]:

# Kiểm tra nhanh các thư viện cơ bản (không bắt buộc tất cả phải có)
import sys, json, math, random, os, time
import numpy as np

try:
    import sklearn
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, confusion_matrix
    SKLEARN_OK = True
except Exception as e:
    SKLEARN_OK = False
    print("Thiếu scikit-learn. Các phần dùng sklearn sẽ không chạy:", e)

try:
    import networkx as nx
    NETWORKX_OK = True
except Exception as e:
    NETWORKX_OK = False
    print("Thiếu networkx. Các phần đồ thị sẽ không chạy:", e)

# Các phần tùy chọn (có thể không có internet nên sẽ fail ở đây, cứ để False nếu không)
try:
    import torch
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False

try:
    import transformers
    TRANSFORMERS_OK = True
except Exception as e:
    TRANSFORMERS_OK = False

print("SKLEARN_OK =", SKLEARN_OK, "| NETWORKX_OK =", NETWORKX_OK, "| TORCH_OK =", TORCH_OK, "| TRANSFORMERS_OK =", TRANSFORMERS_OK)


SKLEARN_OK = True | NETWORKX_OK = True | TORCH_OK = True | TRANSFORMERS_OK = True



## 9) ViT (Vision Transformer) – Ứng dụng: **Phát hiện sản phẩm lỗi vs đạt** (demo)

**Mục tiêu thực tế:** Trong dây chuyền sản xuất, ta muốn kiểm tra nhanh ảnh một sản phẩm là *đạt* hay *lỗi*.
Ở đây ta sẽ demo 2 cách:

1. **Tối giản, ngoại tuyến:** Trích xuất **patch embedding kiểu ViT** bằng `numpy` (không train Transformer),
   sau đó huấn luyện **Logistic Regression** để phân loại ảnh *giả lập* (hoa văn/sọc v.v.). Mục tiêu: hiểu pipeline patchify → embedding → phân loại.

2. **Tùy chọn (cần internet):** Dùng ViT pre-trained (`google/vit-base-patch16-224` qua `transformers`) để inference ảnh thật
   (bạn có thể tải hình, hoặc gắn trực tiếp ảnh tại chỗ). Đây là cách thực tế khi có model đã học trước.


In [3]:

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ---- Tạo bộ dữ liệu ảnh "giả lập" (64x64 RGB) gồm 2 lớp: 'dat' (0) và 'loi' (1)
# Ý tưởng: lớp 0 có hoa văn 'mịn', lớp 1 có sọc/đốm rõ -> tạo khác biệt cấu trúc cục bộ để patch thấy được.

def make_texture_ok(n=200, size=64, seed=0):
    rng = np.random.default_rng(seed)
    imgs = rng.normal(127, 20, size=(n, size, size, 3)).clip(0,255).astype(np.uint8)
    # Làm mịn nhẹ
    for i in range(n):
        imgs[i] = (imgs[i].astype(np.float32)*0.9 + 12).clip(0,255).astype(np.uint8)
    return imgs

def make_texture_ng(n=200, size=64, seed=1):
    rng = np.random.default_rng(seed)
    imgs = rng.normal(127, 20, size=(n, size, size, 3)).clip(0,255).astype(np.uint8)
    # Thêm sọc mạnh theo trục x
    for i in range(n):
        for x in range(0, size, 4):
            imgs[i, :, x:x+2, :] = 255
    # Thêm đốm nhiễu
    mask = rng.random((n, size, size)) < 0.02
    imgs[mask, :] = 0
    return imgs

X_ok = make_texture_ok(250, 64, seed=42)
X_ng = make_texture_ng(250, 64, seed=43)

X = np.concatenate([X_ok, X_ng], axis=0)
y = np.array([0]*len(X_ok) + [1]*len(X_ng))  # 0: đạt, 1: lỗi

# ---- Patchify: chia ảnh 64x64 thành các patch 8x8 (-> 8x8 = 64 patch). Flatten mỗi patch.
def patchify(img, patch=8):
    H,W,C = img.shape
    assert H%patch==0 and W%patch==0
    patches = []
    for i in range(0, H, patch):
        for j in range(0, W, patch):
            p = img[i:i+patch, j:j+patch, :].reshape(-1)
            patches.append(p)
    return np.stack(patches, axis=0)  # [num_patches, patch*patch*C]

def vit_style_embed(imgs, patch=8, proj_dim=64, seed=0):
    # Mô phỏng ViT embedding: flatten patch -> (linear projection) -> mean pooling toàn bộ patch embedding
    rng = np.random.default_rng(seed)
    P = patch*patch*3
    W = rng.normal(0, 1/np.sqrt(P), size=(P, proj_dim)).astype(np.float32)  # ma trận chiếu ngẫu nhiên cố định
    feats = []
    for img in imgs:
        patches = patchify(img, patch=patch)  # [num_patches, P]
        emb = patches @ W                      # [num_patches, proj_dim]
        cls_like = emb.mean(axis=0)            # giống token [CLS] = trung bình
        feats.append(cls_like)
    return np.stack(feats, axis=0)             # [N, proj_dim]

X_feat = vit_style_embed(X, patch=8, proj_dim=128, seed=123)
X_tr, X_te, y_tr, y_te = train_test_split(X_feat, y, test_size=0.3, random_state=0, stratify=y)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

print(classification_report(y_te, y_pred, target_names=["dat","loi"]))


              precision    recall  f1-score   support

         dat       1.00      1.00      1.00        75
         loi       1.00      1.00      1.00        75

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150



In [4]:

# (Tùy chọn) ViT thật với transformers + torch. Cần internet để pip install và tải model.
# Nếu đã có sẵn môi trường DL, bỏ qua lệnh cài đặt.
# Sử dụng cho bài toán: zero-shot / inference nhanh với model pre-trained.

# !!! Chỉ chạy khi có internet !!!
# %pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# %pip install -U transformers timm pillow

USE_REAL_VIT = False  # đổi thành True nếu bạn đã cài thành công

if USE_REAL_VIT:
    import torch
    from PIL import Image
    from transformers import AutoImageProcessor, ViTForImageClassification

    model_name = "google/vit-base-patch16-224"
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = ViTForImageClassification.from_pretrained(model_name)

    # Tải ảnh mẫu (bạn thay đường dẫn ảnh của bạn vào đây)
    # img = Image.open("your_image.jpg").convert("RGB")

    # Demo dùng ảnh tổng hợp từ mảng (chỉ để code không lỗi nếu bạn chưa có ảnh)
    import numpy as np
    arr = np.random.randint(0, 255, (224,224,3), dtype=np.uint8)
    img = Image.fromarray(arr)

    inputs = processor(images=img, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = logits.argmax(-1).item()
    label = model.config.id2label[pred]
    print("Dự đoán:", label)
else:
    print("Để chạy ViT thật: bật USE_REAL_VIT=True sau khi đã cài torch/transformers.")


Để chạy ViT thật: bật USE_REAL_VIT=True sau khi đã cài torch/transformers.



## 10) CLIP – Ứng dụng: **Tìm ảnh theo văn bản (Image Search) & Zero-shot Classification** (demo)

**Bối cảnh:** Bạn có một thư mục ảnh sản phẩm, muốn **gõ từ khóa** (ví dụ: "điện thoại đen viền mỏng") để tìm ảnh phù hợp nhanh.

Ta làm 2 hướng:
1. **Mô phỏng ý tưởng CLIP (ngoại tuyến):** trích xuất *đặc trưng ảnh* đơn giản (histogram màu + HOG nhẹ nếu có),
   trích xuất *đặc trưng văn bản* bằng TF-IDF. Quy về cùng không gian qua chuẩn hóa & cosine similarity → **tìm ảnh gần nhất**.
   (Đây không phải CLIP thật, nhưng tái hiện workflow tìm kiếm đa phương thức.)

2. **Tùy chọn (cần internet):** Dùng CLIP pre-trained (`openai/clip-vit-base-patch32`) từ `transformers` để embed ảnh & text,
   sau đó tìm kiếm bằng cosine giống paper gốc.


In [5]:

# Mô phỏng CLIP: tạo 1 "bộ sưu tập" ảnh giả + caption, sau đó truy vấn bằng văn bản
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Giả lập 6 ảnh sản phẩm với 'mô tả' (caption) ngắn
image_ids = [f"img_{i}.jpg" for i in range(6)]
captions = [
    "điện thoại đen viền mỏng camera kép",
    "tai nghe bluetooth màu trắng chống ồn",
    "laptop xám mỏng nhẹ màn 14 inch",
    "điện thoại xanh dương ba camera",
    "máy ảnh đen ống kính lớn",
    "điện thoại đen viền dày pin trâu"
]

# Text encoder (mô phỏng) bằng TF-IDF
vectorizer = TfidfVectorizer(ngram_range=(1,2))
text_embs = vectorizer.fit_transform(captions)   # shape [6, V]

def query_search(query, topk=3):
    q = vectorizer.transform([query])
    sims = cosine_similarity(q, text_embs)[0]
    idx = np.argsort(-sims)[:topk]
    return [(image_ids[i], captions[i], float(sims[i])) for i in idx]

print("Query: 'điện thoại đen viền mỏng'")
for iid, cap, sc in query_search("điện thoại đen viền mỏng"):
    print(f"- {iid:8s} | {cap:45s} | sim={sc:.3f}")


Query: 'điện thoại đen viền mỏng'
- img_0.jpg | điện thoại đen viền mỏng camera kép           | sim=0.777
- img_5.jpg | điện thoại đen viền dày pin trâu              | sim=0.527
- img_3.jpg | điện thoại xanh dương ba camera               | sim=0.201


In [6]:

# (Tùy chọn) CLIP thật qua transformers
# %pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# %pip install -U transformers pillow

USE_REAL_CLIP = False  # bật True nếu đã cài thành công

if USE_REAL_CLIP:
    import torch
    from PIL import Image
    from transformers import CLIPProcessor, CLIPModel

    model_name = "openai/clip-vit-base-patch32"
    model = CLIPModel.from_pretrained(model_name)
    processor = CLIPProcessor.from_pretrained(model_name)

    # Ví dụ: 3 ảnh (bạn thay bằng ảnh thật của bạn)
    imgs = [Image.fromarray(np.random.randint(0,255,(224,224,3),dtype=np.uint8)) for _ in range(3)]
    texts = ["điện thoại đen", "tai nghe trắng", "laptop mỏng nhẹ"]

    inputs = processor(text=texts, images=imgs, return_tensors="pt", padding=True)
    with torch.no_grad():
        out = model(**inputs)
        i_emb = out.image_embeds / out.image_embeds.norm(p=2, dim=-1, keepdim=True)
        t_emb = out.text_embeds / out.text_embeds.norm(p=2, dim=-1, keepdim=True)
        sims = (t_emb @ i_emb.T).cpu().numpy()  # [len(texts), len(imgs)]

    print("Ma trận độ tương đồng (text x image):\n", np.round(sims,3))
else:
    print("Để chạy CLIP thật: bật USE_REAL_CLIP=True sau khi đã cài torch/transformers.")


Để chạy CLIP thật: bật USE_REAL_CLIP=True sau khi đã cài torch/transformers.



## 11) Self-Organizing Maps (SOM) – Ứng dụng: **Phân cụm khách hàng** (Iris ~ mô phỏng RFM)

**Ý tưởng:** SOM ánh xạ dữ liệu đa chiều về lưới 2D, giữ **cấu trúc topological**, hữu ích để **phân khúc khách hàng**.
Ta dùng bộ **Iris** (4 đặc trưng) như proxy dữ liệu RFM (đơn giản).

**Bước:** Chuẩn hóa → Train SOM 10x10 → Gán mỗi điểm vào Best Matching Unit (BMU) → Quan sát cụm.


In [7]:

import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from collections import Counter

# SOM tối giản
class MiniSOM:
    def __init__(self, m, n, dim, lr=0.5, sigma=None, iters=2000, seed=0):
        self.m, self.n, self.dim = m, n, dim
        self.lr0 = lr
        self.sigma0 = sigma if sigma is not None else max(m,n)/2
        self.iters = iters
        rng = np.random.default_rng(seed)
        self.W = rng.normal(0,1,size=(m*n, dim))

    def _grid(self):
        # tọa độ lưới 2D cho mỗi neuron
        coords = np.array([(i,j) for i in range(self.m) for j in range(self.n)])
        return coords

    def _bmu(self, x):
        # tìm BMU (neuron gần nhất theo L2)
        d = ((self.W - x)**2).sum(axis=1)
        return np.argmin(d)

    def fit(self, X):
        coords = self._grid()
        for t in range(1, self.iters+1):
            x = X[np.random.randint(0, len(X))]
            bmu = self._bmu(x)
            # lịch lr & sigma giảm dần
            lr = self.lr0 * np.exp(-t/self.iters)
            sigma = self.sigma0 * np.exp(-t/self.iters)
            # ảnh hưởng theo khoảng cách lưới
            d2 = ((coords - coords[bmu])**2).sum(axis=1)
            h = np.exp(-d2/(2*sigma**2))  # [m*n]
            # cập nhật trọng số
            self.W += lr * h[:,None] * (x - self.W)

    def predict_bmu(self, X):
        return np.array([self._bmu(x) for x in X])

# Load dữ liệu
iris = load_iris()
X = iris['data']
y = iris['target']
sc = StandardScaler()
Xz = sc.fit_transform(X)

som = MiniSOM(m=10, n=10, dim=Xz.shape[1], lr=0.5, iters=3000, seed=42)
som.fit(Xz)
bmu_idx = som.predict_bmu(Xz)

# Thống kê mỗi cụm (BMU) chứa nhãn nào
counts = {}
for bmu, label in zip(bmu_idx, y):
    counts.setdefault(bmu, []).append(label)

cluster_summ = {k: dict(Counter(v)) for k,v in counts.items()}
# In top 10 BMU đông nhất
top = sorted(cluster_summ.items(), key=lambda kv: sum(kv[1].values()), reverse=True)[:10]
print("Top 10 BMU (neuron) đông nhất & phân bố nhãn:")
for nid, dist in top:
    print(f"BMU {nid:3d}: total={sum(dist.values()):3d} | {dist}")


Top 10 BMU (neuron) đông nhất & phân bố nhãn:
BMU  39: total=  9 | {np.int64(0): 9}
BMU   3: total=  7 | {np.int64(0): 7}
BMU  66: total=  6 | {np.int64(1): 6}
BMU  50: total=  6 | {np.int64(2): 6}
BMU  90: total=  6 | {np.int64(2): 6}
BMU  29: total=  5 | {np.int64(0): 5}
BMU  70: total=  5 | {np.int64(2): 5}
BMU   7: total=  4 | {np.int64(0): 4}
BMU   9: total=  4 | {np.int64(0): 4}
BMU   4: total=  4 | {np.int64(0): 4}



## 12) GNN (Graph Neural Networks) – Ứng dụng: **Phát hiện cộng đồng** (Karate Club)

**Bối cảnh:** Mạng xã hội (graph) – ta muốn dự đoán **cộng đồng** của mỗi người dùng.
Ở đây ta dùng đồ thị **Zachary Karate Club** kinh điển.

Ta xây một **GNN tổng quát kiểu GraphSAGE (mean)**:
- Mỗi lớp: `h_v^{(k+1)} = MLP( concat( h_v^{(k)}, mean_{u∈N(v)} h_u^{(k)} ) )`
- Train bằng cross-entropy từ vài **nhãn mồi** (semi-supervised).

Mục tiêu: thấy quy trình GNN cơ bản mà **không phụ thuộc framework DL**.


In [8]:

import numpy as np

if not 'NETWORKX_OK' in globals() or not NETWORKX_OK:
    print("Cần networkx cho ví dụ này.")
else:
    import networkx as nx

    G = nx.karate_club_graph()
    n = G.number_of_nodes()
    A = nx.to_numpy_array(G)  # adjacency (0/1)

    # Nhãn ground truth theo club ('Mr. Hi' vs 'Officer')
    clubs = np.array([0 if G.nodes[i]['club'] == 'Mr. Hi' else 1 for i in range(n)])

    rng = np.random.default_rng(0)
    # Đặc trưng khởi tạo (d_x = 16)
    X = rng.normal(0,1,size=(n,16)).astype(np.float32)

    # Chia tập train (10 node gán nhãn), val/test còn lại
    idx_all = np.arange(n)
    rng.shuffle(idx_all)
    idx_train = idx_all[:10]
    idx_test  = idx_all[10:]

    # GraphSAGE (mean) 2 lớp, numpy thuần
    def mean_agg(H, A):
        deg = A.sum(axis=1, keepdims=True) + 1e-8
        return (A @ H) / deg

    def relu(z): return np.maximum(z,0)
    def softmax(z):
        z = z - z.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / e.sum(axis=1, keepdims=True)

    # Tham số
    d_in, d_h, d_out = X.shape[1], 32, 2
    W1 = rng.normal(0, 0.1, size=(d_in*2, d_h))
    b1 = np.zeros((d_h,))
    W2 = rng.normal(0, 0.1, size=(d_h*2, d_out))
    b2 = np.zeros((d_out,))

    lr = 0.05
    epochs = 400

    def forward(X):
        N = X.shape[0]
        m1 = mean_agg(X, A)                      # [N, d_in]
        H1 = relu( X @ W1[:d_in] + m1 @ W1[d_in:] + b1 )  # concat bằng phép cộng tuyến tính tương đương
        m2 = mean_agg(H1, A)
        Z  = ( H1 @ W2[:d_h] + m2 @ W2[d_h:] + b2 )
        P  = softmax(Z)
        return H1, P

    def loss_and_grads(X, labels, idx):
        # cross-entropy trên idx
        N = X.shape[0]
        H1, P = forward(X)

        Y = np.zeros((N, d_out))
        Y[np.arange(N), labels] = 1.0

        mask = np.zeros((N,1))
        mask[idx] = 1.0
        L = - (mask * (Y * np.log(P+1e-9)).sum(axis=1, keepdims=True)).sum() / mask.sum()

        # Backprop (tối giản, không quá tối ưu; minh họa thôi)
        # dZ
        dZ = (P - Y) * mask  # [N,d_out]
        m2 = mean_agg(H1, A)

        # grads W2,b2
        dW2a = H1.T @ dZ      # phần với W2[:d_h]
        dW2b = m2.T @ dZ      # phần với W2[d_h:]
        db2  = dZ.sum(axis=0)

        # dH1 từ nhánh Z
        dH1 = dZ @ W2[:d_h].T
        dm2 = dZ @ W2[d_h:].T
        # dm2 = d(mean_agg(H1)) -> phân bố lại về H1
        # mean_agg: m2_i = sum_j A_ij H1_j / deg_i
        # gradient ngược: dH1_j += sum_i (A_ij/deg_i) * dm2_i
        deg = A.sum(axis=1, keepdims=True) + 1e-8
        back = (dm2 / deg)  # [N,d_h]
        dH1 += A.T @ back

        # ReLU
        dH1[H1<=0] = 0

        # Lớp 1
        m1 = mean_agg(X, A)
        dW1a = X.T @ dH1     # cho W1[:d_in]
        dW1b = m1.T @ dH1    # cho W1[d_in:]
        db1  = dH1.sum(axis=0)

        # Không cập nhật X (đặc trưng ngõ vào cố định)
        return float(L), dW2a, dW2b, db2, dW1a, dW1b, db1

    for ep in range(1, epochs+1):
        L, dW2a, dW2b, db2, dW1a, dW1b, db1 = loss_and_grads(X, clubs, idx_train)
        W2[:d_h] -= lr * dW2a; W2[d_h:] -= lr * dW2b; b2 -= lr * db2
        W1[:d_in] -= lr * dW1a; W1[d_in:] -= lr * dW1b; b1 -= lr * db1
        if ep % 50 == 0:
            _, P = forward(X)
            pred = P.argmax(axis=1)
            acc_train = (pred[idx_train] == clubs[idx_train]).mean()
            acc_test  = (pred[idx_test]  == clubs[idx_test]).mean()
            print(f"Epoch {ep:3d} | loss={L:.3f} | acc_train={acc_train:.2f} | acc_test={acc_test:.2f}")


Epoch  50 | loss=0.004 | acc_train=1.00 | acc_test=0.96
Epoch 100 | loss=0.001 | acc_train=1.00 | acc_test=0.92
Epoch 150 | loss=0.001 | acc_train=1.00 | acc_test=0.92
Epoch 200 | loss=0.001 | acc_train=1.00 | acc_test=0.92
Epoch 250 | loss=0.000 | acc_train=1.00 | acc_test=0.92
Epoch 300 | loss=0.000 | acc_train=1.00 | acc_test=0.92
Epoch 350 | loss=0.000 | acc_train=1.00 | acc_test=0.92
Epoch 400 | loss=0.000 | acc_train=1.00 | acc_test=0.92



## 13) GCN (Graph Convolutional Networks) – Ứng dụng: **Phân loại node bán giám sát** (Karate Club)

**Công thức Kipf & Welling (2017):**  
\(
\hat{A} = A + I,\quad \hat{D}_{ii} = \sum_j \hat{A}_{ij},\quad \tilde{A} = \hat{D}^{-1/2}\hat{A}\hat{D}^{-1/2}
\)  
\(
H^{(1)} = \mathrm{ReLU}(\tilde{A} X W^{(0)}),\quad
Z = \tilde{A} H^{(1)} W^{(1)}
\)

Ta dùng cùng đồ thị Karate, train với 10 node gán nhãn.


In [9]:
import numpy as np

if not 'NETWORKX_OK' in globals() or not NETWORKX_OK:
    print("Cần networkx cho ví dụ này.")
else:
    import networkx as nx

    G = nx.karate_club_graph()
    n = G.number_of_nodes()
    A = nx.to_numpy_array(G)
    I = np.eye(n)
    A_hat = A + I
    D_hat = np.diag( A_hat.sum(axis=1) )
    D_inv_sqrt = np.linalg.inv(np.sqrt(D_hat + 1e-8*np.eye(n)))
    A_tilde = D_inv_sqrt @ A_hat @ D_inv_sqrt    # normalized adjacency

    # labels
    labels = np.array([0 if G.nodes[i]['club']=='Mr. Hi' else 1 for i in range(n)])

    rng = np.random.default_rng(123)
    X = rng.normal(0,1,size=(n, 16)).astype(np.float32)

    idx_all = np.arange(n)
    rng.shuffle(idx_all)
    idx_train = idx_all[:10]
    idx_test  = idx_all[10:]

    d_in, d_h, d_out = X.shape[1], 32, 2
    W0 = rng.normal(0, 0.1, size=(d_in, d_h))
    W1 = rng.normal(0, 0.1, size=(d_h, d_out))

    def relu(z): return np.maximum(z,0)
    def softmax(z):
        z = z - z.max(axis=1, keepdims=True); e = np.exp(z)
        return e / e.sum(axis=1, keepdims=True)

    def forward(X):
        H1 = relu(A_tilde @ X @ W0)   # GCN layer 1
        Z  = A_tilde @ H1 @ W1        # GCN layer 2
        P  = softmax(Z)
        return H1, P

    def loss_and_grads(X, labels, idx):
        H1, P = forward(X)
        Y = np.zeros((n, d_out)); Y[np.arange(n), labels] = 1.0

        mask = np.zeros((n,1)); mask[idx] = 1.0
        L = - (mask * (Y * np.log(P+1e-9)).sum(axis=1, keepdims=True)).sum() / mask.sum()

        # Backprop
        dZ = (P - Y) * mask  # [n, d_out]
        dW1 = (H1.T @ (A_tilde.T @ dZ))
        dH1 = (A_tilde.T @ dZ) @ W1.T
        dH1[H1<=0] = 0
        dW0 = (X.T @ (A_tilde.T @ dH1))

        return float(L), dW0, dW1

    lr = 0.1
    for ep in range(1, 401):
        L, dW0, dW1 = loss_and_grads(X, labels, idx_train)
        W0 -= lr * dW0; W1 -= lr * dW1
        if ep % 50 == 0:
            _, P = forward(X)
            pred = P.argmax(axis=1)
            acc_train = (pred[idx_train] == labels[idx_train]).mean()
            acc_test  = (pred[idx_test]  == labels[idx_test]).mean()
            print(f"Epoch {ep:3d} | loss={L:.3f} | acc_train={acc_train:.2f} | acc_test={acc_test:.2f}")


Epoch  50 | loss=0.019 | acc_train=1.00 | acc_test=0.88
Epoch 100 | loss=0.007 | acc_train=1.00 | acc_test=0.88
Epoch 150 | loss=0.004 | acc_train=1.00 | acc_test=0.88
Epoch 200 | loss=0.003 | acc_train=1.00 | acc_test=0.88
Epoch 250 | loss=0.002 | acc_train=1.00 | acc_test=0.88
Epoch 300 | loss=0.001 | acc_train=1.00 | acc_test=0.88
Epoch 350 | loss=0.001 | acc_train=1.00 | acc_test=0.88
Epoch 400 | loss=0.001 | acc_train=1.00 | acc_test=0.88



---

### Ghi chú nộp bài
- Chụp kết quả training (accuracy) cho phần GNN/GCN.
- Với ViT/CLIP: nếu có internet, hãy bật cell **USE_REAL_VIT/USE_REAL_CLIP = True** và thử với ảnh thật của bạn.
- Trình bày ngắn gọn kết luận: mỗi mô hình phù hợp cho bài toán nào, ưu/nhược, đề xuất khi triển khai thực tế.

**Chúc bạn học tốt!** 💪
